In [10]:
import json
import uuid
import boto3
import pytz
from datetime import datetime, timedelta
from dotenv import load_dotenv
from typing import Annotated, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

load_dotenv(override=True)

# ── CloudWatch client ────────────────────────────────────────────────────────
cloudwatch_client = boto3.client("cloudwatch")


# ── Tool definition ──────────────────────────────────────────────────────────
@tool
def get_cpu_metrics(
    instanceId: str,
    Instance: str = "AWS/EC2",
    Metric: str = "CPUUtilization",
    last_hours: int = 6,
    period: int = 1800,
) -> dict:
    """
    Fetches CPUUtilization metrics for the user-provided instance
    over the last N hours with configurable period intervals.

    Args:
        instanceId: EC2 instance ID (e.g. i-0327bf109dc40d412)
        Instance:   CloudWatch namespace  (default: AWS/EC2)
        Metric:     Metric name           (default: CPUUtilization)
        last_hours: Look-back window in hours  (default: 6)
        period:     Aggregation period in seconds (default: 1800 = 30 min)
    """
    end_time = datetime.now(pytz.utc)
    start_time = end_time - timedelta(hours=last_hours)

    response = cloudwatch_client.get_metric_statistics(
        Namespace=Instance,
        MetricName=Metric,
        Dimensions=[{"Name": "InstanceId", "Value": instanceId}],
        StartTime=start_time,
        EndTime=end_time,
        Period=period,
        Statistics=["Average"],
        Unit="Percent",
    )

    ist = pytz.timezone("Asia/Kolkata")
    for point in response["Datapoints"]:
        point["Timestamp"] = (
            point["Timestamp"].astimezone(ist).strftime("%Y-%m-%d %H:%M:%S IST")
        )

    response["Datapoints"].sort(key=lambda x: x["Timestamp"], reverse=True)
    return {"Datapoints": response["Datapoints"]}


tools = [get_cpu_metrics]
tools_by_name = {t.name: t for t in tools}


# ── Graph state ──────────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


# ── LLM with tools bound ─────────────────────────────────────────────────────
llm = ChatOpenAI(model="gpt-4o", temperature=0)
llm_with_tools = llm.bind_tools(tools)

SYSTEM_PROMPT = "You are a cloud watcher agent that monitors cloud infrastructure."


# ── Graph nodes ──────────────────────────────────────────────────────────────
def call_llm(state: AgentState) -> AgentState:
    """Send messages to the LLM (with tools available)."""
    messages = [SystemMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}


def execute_tools(state: AgentState) -> AgentState:
    """Execute every tool call requested by the last AI message."""
    last_message = state["messages"][-1]
    tool_messages = []

    for tool_call in last_message.tool_calls:
        tool_fn = tools_by_name[tool_call["name"]]
        result = tool_fn.invoke(tool_call["args"])
        tool_messages.append(
            ToolMessage(
                content=json.dumps(result, default=str),
                tool_call_id=tool_call["id"],
            )
        )

    return {"messages": tool_messages}


# ── Routing logic ────────────────────────────────────────────────────────────
def should_continue(state: AgentState) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "execute_tools"
    return END


# ── Build the graph ──────────────────────────────────────────────────────────
def build_graph():
    memory = MemorySaver()                          # ← checkpoint store

    graph = StateGraph(AgentState)

    graph.add_node("call_llm", call_llm)
    graph.add_node("execute_tools", execute_tools)

    graph.set_entry_point("call_llm")

    graph.add_conditional_edges(
        "call_llm",
        should_continue,
        {"execute_tools": "execute_tools", END: END},
    )

    graph.add_edge("execute_tools", "call_llm")

    return graph.compile(checkpointer=memory)       # ← pass checkpointer here


# ── Helper: single turn ───────────────────────────────────────────────────────
def chat(agent, thread_id: str, user_message: str) -> str:
    """
    Send one user message and return the agent's reply.
    The same thread_id preserves conversation history across turns.
    """
    config = {"configurable": {"thread_id": thread_id}}
    result = agent.invoke(
        {"messages": [HumanMessage(content=user_message)]},
        config=config,
    )
    return result["messages"][-1].content

def main():
    agent = build_graph()
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
 
    def run(query: str) -> str:
        final_state = agent.invoke(
            {"messages": [HumanMessage(content=query)]},
            config=config,
        )
        return final_state["messages"][-1].content
 
    # ── Test queries ─────────────────────────────────────────────────────────
 
    # 1. Basic CPU fetch — last 20 hours, 30-min buckets
    q1 = (
        "Give observations on CPU utilization for ec2 instance over last 20 hours "
        "with 30 mins timeline in detail for instanceId i-0327bf109dc40d412"
    )
 
    # 2. Memory test — relies on the answer from q1
    q2 = "What was the peak CPU value you just reported and at what time?"
 
    # 3. Shorter window — last 3 hours, 10-min buckets (period=600)
    q3 = (
        "Check CPU for instanceId i-0327bf109dc40d412 over the last 3 hours "
        "with a 10 minute interval"
    )

    q1_response = run(q1)
    print(q1_response)

    q2_response = run(q2)
    print(q2_response)

main()

Here are the observations on CPU utilization for the EC2 instance with ID `i-0327bf109dc40d412` over the last 20 hours, with data points collected every 30 minutes:

1. **May 18, 2026, 17:06 IST**: The CPU utilization was at 0.27%.
2. **May 18, 2026, 17:36 IST**: The CPU utilization slightly decreased to 0.27%.
3. **May 18, 2026, 18:06 IST**: There was a slight increase in CPU utilization to 0.30%.
4. **May 18, 2026, 18:36 IST**: The CPU utilization slightly decreased to 0.28%.
5. **May 18, 2026, 19:06 IST**: The CPU utilization was recorded at 0.27%.
6. **May 19, 2026, 12:06 IST**: A significant drop in CPU utilization was observed, down to 0.01%.
7. **May 19, 2026, 12:36 IST**: The CPU utilization increased to 0.38%.

Overall, the CPU utilization remained relatively low, with a notable drop and subsequent increase observed on May 19, 2026. The utilization values indicate that the instance was not heavily loaded during this period.
The peak CPU utilization value reported was **0.38%**